# NBA Chat with Data — AI-Powered Basketball Analytics

Launch the same **Chainlit** chat app from `apps/chat` inside a notebook-friendly workflow to explore 160+ tables of NBA data using natural language, SQL, Python, and interactive visualizations.

**Features:**
- 7 LLM providers (OpenAI, Anthropic, Google, Ollama, LM Studio, GitHub Copilot, Custom)
- Interactive Plotly & matplotlib charts with team colors
- Advanced metrics (TS%, eFG%, Usage Rate, Pace, Net Rating)
- SQL sandbox + Python execution environment
- Export results as CSV, XLSX, or JSON with one click
- Copy SQL/Python code from any step, or export full session as a script
- Web search for latest NBA news
- Settings panel, chat profiles, action buttons

**Requirements:** Local browser access to this notebook environment and your own API key from any supported provider.

**Note:** This notebook is a local-only launcher for the shared chat app. It does not create a public tunnel.

---

In [ ]:
# Cell 1: Resolve chat checkout and install dependencies
import subprocess
import sys
import tomllib
from pathlib import Path

REPO_URL = "https://github.com/wyattowalsh/nbadb.git"
REPO_REF = "af26ef80a1e6852991fae72376ce4694e6732fbf"
IN_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path.cwd().resolve()


def _find_checked_out_chat_dir() -> Path | None:
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidate = base / "apps" / "chat"
        if (candidate / "chainlit_app.py").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    return None


def _clone_pinned_repo() -> Path:
    clone_dir = WORK_DIR / f"nbadb-{REPO_REF[:8]}"
    chat_dir = clone_dir / "apps" / "chat"
    if (chat_dir / "chainlit_app.py").exists() and (chat_dir / "pyproject.toml").exists():
        print(f"Using existing pinned fallback clone at {chat_dir}")
        return chat_dir
    if clone_dir.exists():
        raise FileExistsError(f"{clone_dir} already exists but does not look like the chat app checkout")

    print(f"Cloning nbadb repository at commit {REPO_REF}...")
    subprocess.check_call(
        ["git", "clone", REPO_URL, str(clone_dir)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ["git", "-C", str(clone_dir), "checkout", "--detach", REPO_REF],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"Repository cloned and pinned to {REPO_REF}.")
    return chat_dir


CHAT_DIR = _find_checked_out_chat_dir() or _clone_pinned_repo()
PYPROJECT_PATH = CHAT_DIR / "pyproject.toml"

with PYPROJECT_PATH.open("rb") as handle:
    project = tomllib.load(handle)["project"]

deps = list(project["dependencies"])
for extra in project.get("optional-dependencies", {}).values():
    deps.extend(extra)
deps = list(dict.fromkeys(deps))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *deps],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
 )
print(f"Using chat app at {CHAT_DIR}")
print(f"Installed {len(deps)} dependencies from {PYPROJECT_PATH}")

In [ ]:
# Cell 2: Configure provider + API key
import getpass
import json
from pathlib import Path

PROVIDER = "openai"   # Change to: "anthropic", "google", "ollama", "lmstudio", "copilot", "custom"
MODEL = "gpt-4.1"     # Change to a provider-specific model ID, for example "claude-sonnet-4" or "gemini-2.5-flash"

_KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "google": "GOOGLE_API_KEY",
}
_KEY_OPTIONAL_PROVIDERS = {"ollama", "lmstudio", "copilot"}

api_key = None
if PROVIDER not in _KEY_OPTIONAL_PROVIDERS:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets_client = UserSecretsClient()
        api_key = secrets_client.get_secret(_KEY_NAMES.get(PROVIDER, "API_KEY"))
        print(f"Loaded {PROVIDER} key from Kaggle secrets")
    except Exception:
        api_key = getpass.getpass(f"Enter your {PROVIDER} API key: ") or None

config_dir = Path.home() / ".nbadb"
config_dir.mkdir(parents=True, exist_ok=True)
config = {"provider": PROVIDER, "model": MODEL}
if api_key:
    config["api_key"] = api_key
(config_dir / "chat.json").write_text(json.dumps(config))

print(f"Config saved: provider={PROVIDER}, model={MODEL}")
if api_key:
    print("API key stored in ~/.nbadb/chat.json for this notebook session.")
else:
    print("No API key stored. This is expected for Ollama, LM Studio, or Copilot.")
print("You can still override provider, model, or key in the Chainlit settings panel.")

In [ ]:
# Cell 3: Ensure NBA database is available
import shutil
from pathlib import Path

# Kaggle attaches the dataset at /kaggle/input/basketball/
KAGGLE_DB = Path("/kaggle/input/basketball/nba.duckdb")
LOCAL_DB = Path.home() / ".nbadb/data/nba.duckdb"

if KAGGLE_DB.exists() and not LOCAL_DB.exists():
    LOCAL_DB.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(KAGGLE_DB, LOCAL_DB)
    print(f"Copied database to {LOCAL_DB}")
elif LOCAL_DB.exists():
    print(f"Database ready at {LOCAL_DB}")
else:
    print("Database not found — will download from Kaggle on first chat")

In [ ]:
# Cell 4: Enter the chat app directory
import os
from pathlib import Path

os.chdir(str(CHAT_DIR))
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 5: Launch Chainlit locally with readiness checks
import os
import subprocess
import sys
import time
from pathlib import Path

import httpx
from IPython.display import HTML, display

PORT = 8421
HOST = "127.0.0.1"
BASE_URL = f"http://{HOST}:{PORT}"


def _run_chainlit():
    """Run Chainlit server in the background."""
    env = dict(os.environ)
    return subprocess.Popen(
        [
            sys.executable,
            "-m",
            "chainlit",
            "run",
            "chainlit_app.py",
            "--host",
            HOST,
            "--port",
            str(PORT),
        ],
        cwd=str(CHAT_DIR),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        env=env,
    )


def _wait_for_chainlit(process, url: str, timeout: float = 120.0, interval: float = 1.0) -> None:
    deadline = time.monotonic() + timeout
    last_error: Exception | None = None
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"Chainlit exited early with code {process.returncode}")
        try:
            response = httpx.get(url, timeout=2.0)
            if response.status_code == 200:
                print(f"Chainlit ready at {url}")
                return
            last_error = RuntimeError(f"Unexpected HTTP {response.status_code}")
        except httpx.RequestError as exc:
            last_error = exc
        time.sleep(interval)
    raise TimeoutError(
        f"Chainlit did not become ready at {url} within {timeout:.0f}s. Last error: {last_error}"
    )


process = _run_chainlit()
_wait_for_chainlit(process, BASE_URL)

if Path("/kaggle").exists() or "google.colab" in sys.modules:
    print(
        "This notebook launcher is local-only. If your environment does not expose localhost, "
        "run it in a local notebook session or use your platform's port forwarding."
    )

display(
    HTML(
        f'<h2 style="color:#1D428A">'
        f'Open NBA Chat: <a href="{BASE_URL}" target="_blank">{BASE_URL}</a>'
        f'</h2>'
    )
)

## Usage

1. Click the local link above to open the chat interface
2. Use the **gear icon** to switch providers, models, and temperature
3. Pick a **profile** (Quick Stats, Deep Analysis, Visualization)
4. Ask anything about NBA data!

### Example questions

- "Who are the top 10 scorers this season? Show a bar chart with team colors."
- "Compare LeBron and Curry's career TS% trends"
- "What's the correlation between pace and offensive rating?"
- "Plot a scatter of usage rate vs points for all 20+ PPG scorers"
- "Analyze the top 5 MVP candidates across all advanced metrics"

### Troubleshooting

| Issue | Fix |
| --- | --- |
| Local link unreachable | Run the notebook in a local environment or use your platform's port forwarding |
| API key error | Re-run Cell 2 with the correct key |
| Database missing | Ensure the `wyattowalsh/basketball` dataset is attached |
| Import errors | Re-run Cell 1 to reinstall dependencies |

---

*Built with [nbadb](https://github.com/wyattowalsh/nbadb) — the open-source NBA data platform.*